## IPS workstation

You can use this notebook to quickly generate a tarfile with desired runids for download. 

In [ ]:
# Notebook template, the IPS Framework will add a cell before this one
# defining DATA_FILES as a list of state file paths.

# In this example, this notebook is only generated at the end of the run.

# Bokeh is installed by default on the NERSC Jupyter environment, so works well for this example.
from bokeh.io import output_notebook
from bokeh.plotting import figure, show
import json
from collections import defaultdict

output_notebook()

from api_v1 import get_data_from_runids

DATA_FILES = get_data_from_runids([1,2,3])
mapping = defaultdict(list)

# We will make a few assumptions about the data:
# 1) each runID will have the exact same timestamp keys
# 2) each value for a timestamp key will be an array of an item with "y1", "y2", and "y3" (this item is an example domain science object)

for d in DATA_FILES.values():
    for timestep, files in d.items():
        for f in files:
            with open(f) as json_data:
                mapping[timestep].append(json.load(json_data))

x = sorted(mapping.keys())

COLORS = ['red', 'green', 'blue', 'yellow']
# we will track all y1 changes in one graph, all y2 changes in another graph, etc.
GRAPHS = ['y1', 'y2', 'y3']

for g in GRAPHS:
    graph = figure(title=g)
    graph.xaxis.axis_label = 'IPS Timestep'
    graph.yaxis.axis_label = 'data values'

    y_vals = []

    for v in mapping.values():
        for idx, plot in enumerate(v):
            if idx + 1 > len(y_vals):
                y_vals.append([])
            y_vals[idx].append(plot[g])

    for idx, y in enumerate(y_vals):
        graph.line(x, y, line_color=COLORS[idx % len(COLORS)], line_dash='solid')    
    
    show(graph)
